# Klasifikasi Kematangan Tomat — Data Private, 4 Skenario YOLO26-cls

**Mata Kuliah**: Sains Data Terapan · **Kelompok 2** · **Arsitektur**: YOLO26 (Image Classification)

| # | Skenario | Cara |
|:--|:--|:--|
| 1 | Inference data private dengan pretrained | `yolo26n-cls.pt` tanpa training (+ pemetaan label ImageNet→kelas tomat dari data train) |
| 2 | Fine-tune: ubah hyperparameter, **training head only** | load pretrained, `freeze` semua layer kecuali head, variasi optimizer & `lr0` |
| 3 | Transfer learning: load pretrained, **freeze sebagian layer** | variasi jumlah layer yang di-freeze |
| 4 | Train from scratch | `yolo26n-cls.yaml`, bobot acak |

**Semua skenario memakai split, `imgsz`, `epochs`, `batch`, dan `seed` yang sama** supaya perbandingannya adil.

> Runtime -> Change runtime type -> **T4 GPU** sebelum menjalankan.

**Format data (`.zip`)** — salah satu dari dua:
```
Tomat.zip
|-- mentah/            Tomat.zip
|-- setengah_matang/   |-- train/<kelas>/*.jpg
`-- matang/            |-- val/<kelas>/*.jpg
(auto split 70/15/15)  `-- test/<kelas>/*.jpg
```

## 0. Setup

In [ ]:
import torch
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU tidak aktif! Training akan sangat lambat.')

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# ===== KONFIGURASI (dipakai SEMUA skenario) =====
SEED     = 42
IMGSZ    = 224
EPOCHS   = 30
BATCH    = 32
PATIENCE = 10
PROJECT  = '/content/runs/tomat'
PRETRAINED = 'yolo26n-cls.pt'
SCRATCH    = 'yolo26n-cls.yaml'

## 1. Upload & siapkan dataset private

Kalau data sudah terpisah `train/val/test`, dipakai apa adanya. Kalau hanya folder per kelas, dibagi otomatis **70/15/15** per kelas dengan seed tetap.

> **Hati-hati kebocoran data**: kalau satu buah difoto beberapa kali, foto-foto itu jangan terpisah di train dan test. Pisahkan per buah/sesi, lalu upload dalam format `train/val/test`. Kalau tidak, akurasi akan terlalu optimis.

In [ ]:
from google.colab import files
import os, shutil, random, zipfile
from pathlib import Path

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

print('Pilih file zip dataset private...')
uploaded = files.upload()
zip_name = list(uploaded)[0]

RAW = Path('/content/private_raw')
shutil.rmtree(RAW, ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(RAW)

def clean_dirs(p):
    return sorted(d for d in p.iterdir() if d.is_dir() and not d.name.startswith(('.', '__')))

# masuk ke dalam folder pembungkus (mis. Tomat/Tomat/...) sampai ketemu kelas atau split
root = RAW
while True:
    ds = clean_dirs(root)
    has_imgs = any(f.suffix.lower() in IMG_EXT for f in root.iterdir() if f.is_file())
    if len(ds) == 1 and not has_imgs:
        root = ds[0]
    else:
        break

names = {d.name.lower() for d in clean_dirs(root)}
if {'train', 'val', 'test'} <= names:
    DATA = str(root)
    print('Dataset sudah ter-split, dipakai apa adanya:', DATA)
else:
    DATA = '/content/dataset_private'
    shutil.rmtree(DATA, ignore_errors=True)
    random.seed(SEED)
    for cls_dir in clean_dirs(root):
        imgs = sorted(f for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXT)
        if not imgs:
            print(f'  (lewati {cls_dir.name}: kosong)')
            continue
        random.shuffle(imgs)
        n = len(imgs); c1, c2 = int(n*0.70), int(n*0.85)
        for split, batch in {'train': imgs[:c1], 'val': imgs[c1:c2], 'test': imgs[c2:]}.items():
            out = Path(DATA)/split/cls_dir.name
            out.mkdir(parents=True, exist_ok=True)
            for f in batch:
                shutil.copy(f, out/f.name)
    print('Dataset di-split 70/15/15 ->', DATA)

CLASSES = sorted(d.name for d in clean_dirs(Path(DATA)/'train'))
print('\nKelas:', CLASSES)
print(f'{"split":<6} ' + ' '.join(f'{c:>16}' for c in CLASSES) + '   total')
for split in ['train', 'val', 'test']:
    cnt = [len([f for f in (Path(DATA)/split/c).glob('*') if f.suffix.lower() in IMG_EXT]) for c in CLASSES]
    print(f'{split:<6} ' + ' '.join(f'{n:>16}' for n in cnt) + f'   {sum(cnt)}')

## 2. Helper evaluasi

Semua skenario dievaluasi dengan fungsi yang sama pada **test set**: akurasi top-1, macro-F1, dan prediksi per gambar.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from ultralytics import YOLO

RESULTS = []   # satu baris per eksperimen
PREDS   = {}   # tag -> (y_true, y_pred) untuk confusion matrix

def test_items():
    paths, labels = [], []
    for c in CLASSES:
        for f in sorted((Path(DATA)/'test'/c).glob('*')):
            if f.suffix.lower() in IMG_EXT:
                paths.append(str(f)); labels.append(c)
    return paths, labels

def predict_names(model, paths):
    out = []
    for r in model.predict(paths, imgsz=IMGSZ, verbose=False, stream=True):
        out.append(r.names[r.probs.top1])
    return out

def record(tag, scenario, config, y_true, y_pred, n_trainable=None):
    row = dict(tag=tag, scenario=scenario, config=config,
               top1=accuracy_score(y_true, y_pred),
               macro_f1=f1_score(y_true, y_pred, average='macro', labels=CLASSES, zero_division=0),
               n_test=len(y_true), trainable_params=n_trainable)
    RESULTS.append(row); PREDS[tag] = (y_true, y_pred)
    print(f"[{tag}] top1={row['top1']*100:.2f}%  macro-F1={row['macro_f1']*100:.2f}%  ({config})")
    return row

def evaluate_weights(weights, tag, scenario, config):
    m = YOLO(str(weights))
    paths, y_true = test_items()
    y_pred = predict_names(m, paths)
    return record(tag, scenario, config, y_true, y_pred)

def train_run(tag, scenario, weights, config_desc, **kw):
    model = YOLO(weights)
    model.train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                seed=SEED, project=PROJECT, name=tag, exist_ok=True, plots=True,
                verbose=False, **kw)
    best = Path(model.trainer.save_dir)/'weights'/'best.pt'
    return evaluate_weights(best, tag, scenario, config_desc)

---
## Skenario 1 — Inference data private dengan pretrained model

Bobot `yolo26n-cls.pt` dilatih di ImageNet (1.000 kelas), jadi outputnya label ImageNet, bukan kelas kematangan tomat. Ada dua pengukuran:

- **1a. Langsung**: nama label ImageNet dibandingkan ke nama kelas tomat. Hampir pasti ~0%, karena ruang labelnya berbeda.
- **1b. Dengan pemetaan label**: tiap label ImageNet dipetakan ke kelas tomat mayoritasnya **berdasarkan data train saja**, lalu dievaluasi di test. Ini batas atas yang adil untuk "pretrained tanpa training".

In [ ]:
from collections import Counter, defaultdict

pre = YOLO(PRETRAINED)
test_paths, test_true = test_items()

# 1a. langsung
pred_1a = predict_names(pre, test_paths)
record('s1a_pretrained_langsung', '1. Pretrained inference', 'tanpa training, label ImageNet apa adanya', test_true, pred_1a)

# 1b. pemetaan label dari data train
tr_paths, tr_true = [], []
for c in CLASSES:
    for f in sorted((Path(DATA)/'train'/c).glob('*')):
        if f.suffix.lower() in IMG_EXT:
            tr_paths.append(str(f)); tr_true.append(c)
tr_pred = predict_names(pre, tr_paths)

votes = defaultdict(Counter)
for p, t in zip(tr_pred, tr_true):
    votes[p][t] += 1
mapping = {p: cnt.most_common(1)[0][0] for p, cnt in votes.items()}
default_cls = Counter(tr_true).most_common(1)[0][0]
pred_1b = [mapping.get(p, default_cls) for p in pred_1a]
record('s1b_pretrained_mapping', '1. Pretrained inference', 'label ImageNet -> kelas tomat (mapping dari train)', test_true, pred_1b)

print('\nLabel ImageNet yang paling sering muncul per kelas tomat (test):')
ct = pd.crosstab(pd.Series(test_true, name='kelas asli'), pd.Series(pred_1a, name='prediksi ImageNet'))
for c in CLASSES:
    print(f'  {c:<18}', ct.loc[c].sort_values(ascending=False).head(3).to_dict())

In [ ]:
# contoh visual prediksi pretrained
import random as _r
_r.seed(SEED)
idx = _r.sample(range(len(test_paths)), min(8, len(test_paths)))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(plt.imread(test_paths[i])); ax.axis('off')
    ax.set_title(f'asli: {test_true[i]}\nImageNet: {pred_1a[i]}', fontsize=9)
plt.tight_layout(); plt.show()

---
## Skenario 2 — Fine-tune: ubah hyperparameter, training **head only**

Bobot pretrained dimuat; semua layer **backbone di-freeze** dan hanya head klasifikasi (layer terakhir) yang dilatih. Head otomatis dibuat ulang untuk jumlah kelas tomat. Hyperparameter divariasikan (optimizer dan `lr0`).

In [ ]:
n_layers = len(YOLO(PRETRAINED).model.model)
print('Jumlah modul di model:', n_layers, '(index 0 .. %d, head = index %d)' % (n_layers-1, n_layers-1))
HEAD_FREEZE = n_layers - 1     # freeze index 0..n-2, latih head saja

HP_GRID = [
    dict(optimizer='SGD',   lr0=0.01),
    dict(optimizer='SGD',   lr0=0.001),
    dict(optimizer='AdamW', lr0=0.001),
    dict(optimizer='AdamW', lr0=0.0001),
]
for hp in HP_GRID:
    tag = f"s2_head_{hp['optimizer']}_lr{hp['lr0']}"
    train_run(tag, '2. Fine-tune head only', PRETRAINED,
              f"freeze={HEAD_FREEZE} (head only), {hp['optimizer']}, lr0={hp['lr0']}",
              freeze=HEAD_FREEZE, pretrained=True, **hp)

---
## Skenario 3 — Transfer learning: load pretrained, freeze sebagian layer

Pretrained dimuat, lalu hanya **N layer pertama** yang di-freeze (`freeze=N`), sisanya ikut dilatih. Variasi: tidak freeze (full fine-tune), freeze separuh, dan freeze tiga perempat. Optimizer dan `lr0` sama di semua run (SGD `lr0=0.01`) supaya yang berubah hanya jumlah layer yang di-freeze.

> Poin 3 di soal terpotong logo. Kalau ada instruksi tambahan, tambahkan sendiri di cell ini.

In [ ]:
FREEZE_VARIANTS = {
    'none (full fine-tune)': 0,
    'freeze separuh':        n_layers // 2,
    'freeze 3/4':            (n_layers * 3) // 4,
}
for name, n in FREEZE_VARIANTS.items():
    train_run(f's3_freeze{n}', '3. Transfer learning (freeze sebagian)', PRETRAINED,
              f'freeze={n} ({name}), SGD, lr0=0.01',
              freeze=n, pretrained=True, optimizer='SGD', lr0=0.01)

---
## Skenario 4 — Train from scratch

Arsitektur YOLO26-cls dengan **bobot acak** (tanpa ImageNet), dilatih sepenuhnya di data private. Hyperparameter sama dengan skenario 3.

In [ ]:
train_run('s4_scratch', '4. Train from scratch', SCRATCH,
          'bobot acak, SGD, lr0=0.01',
          pretrained=False, optimizer='SGD', lr0=0.01)

---
## Perbandingan semua skenario

In [ ]:
df = pd.DataFrame(RESULTS)
df['top1 (%)']     = (df.top1*100).round(2)
df['macro-F1 (%)'] = (df.macro_f1*100).round(2)
table = df[['scenario', 'tag', 'config', 'top1 (%)', 'macro-F1 (%)', 'n_test']]
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 200)
display(table)
table.to_csv('/content/hasil_perbandingan.csv', index=False)

plt.figure(figsize=(10, 0.5*len(df)+2))
plt.barh(df.tag[::-1], df.top1[::-1]*100)
plt.xlabel('Top-1 accuracy test (%)'); plt.xlim(0, 100); plt.grid(axis='x', alpha=.3)
for y, v in enumerate(df.top1[::-1]*100):
    plt.text(v+0.5, y, f'{v:.1f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ringkasan terbaik per skenario
best_per = df.loc[df.groupby('scenario')['top1'].idxmax()][['scenario', 'tag', 'config', 'top1 (%)', 'macro-F1 (%)']]
display(best_per)

In [ ]:
# confusion matrix + metrik per kelas untuk model terbaik per skenario
for _, r in best_per.iterrows():
    yt, yp = PREDS[r.tag]
    print('='*70); print(r.scenario, '->', r.tag)
    print(classification_report(yt, yp, labels=CLASSES, zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=CLASSES), display_labels=CLASSES).plot(ax=ax, xticks_rotation=30, colorbar=False)
    ax.set_title(r.tag, fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# kurva training skenario terbaik per skenario (hanya yang pernah di-train)
from IPython.display import Image, display as _d
for tag in best_per.tag:
    p = Path(PROJECT)/tag/'results.png'
    if p.exists():
        print(tag); _d(Image(str(p), width=800))

In [ ]:
# unduh bobot terbaik keseluruhan + tabel hasil
trained = df[df.tag.str.startswith(('s2', 's3', 's4'))]
best_tag = trained.loc[trained.top1.idxmax(), 'tag']
print('Model terbaik (hasil training):', best_tag)
shutil.copy(Path(PROJECT)/best_tag/'weights'/'best.pt', '/content/best_tomat.pt')
files.download('/content/best_tomat.pt')
files.download('/content/hasil_perbandingan.csv')

## Cara membaca hasil

- Bandingkan **top-1 dan macro-F1**. Kalau kelasnya tidak seimbang, macro-F1 lebih jujur daripada top-1.
- Skenario 1a hampir pasti sangat rendah karena beda ruang label, bukan karena fiturnya jelek. 1b adalah pembanding yang adil.
- Yang perlu dilihat: apakah head-only sudah cukup, apakah membuka lebih banyak layer menambah akurasi, dan apakah pretrained mengalahkan scratch pada data kecil.
- Test set kecil membuat selisih 1–2% belum tentu bermakna. Cantumkan jumlah data uji di laporan.